## 데이터 수집 자동화

## 다나와 상품정보 스크래핑

#### 자동 로그인
- 패스워드 숨기기

#### 1. setx 명령어로 시스템에 저장방법
- 패스워드 숨기기
- setx 명령어로 시스템에 저장방법
- powershell

```powershell
> setx DANAWA_PASSWORD "비밀번호"

성공 : 지정한 값을 저장했습니다.

> setx DANAWA_ID "아이디"


성공 : 지정한 값을 저장했습니다.
```

- sysdm.cpl로 시스템 정보에서 환경변수 user 리스트에서 오픈 및 삭제

##### 2. keyring 으로 윈도우 자격증명 저장소

- setx 와 달리 아예 오픈 안됨
- 패키지 설치

In [1]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable

   --------------------------------- ------ 5/6 [keyring]
   ---------------------------------------- 6/6 [keyring]



In [7]:
import keyring
from getpass import getpass

In [3]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호')

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
# 패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장 완료')

아이디/패스워드 저장 완료


In [4]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

'personar'

In [5]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

- 패스워드 오픈 안되고 저장

In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import os

In [9]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://prod.danawa.com/')

wait.until(
    EC.visibility_of_all_element_located((By.ID, 'danawa_footer'))
)

AttributeError: module 'selenium.webdriver.support.expected_conditions' has no attribute 'visibility_of_all_element_located'

In [11]:
# 로그인 링크 확인 후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [12]:
# 로그인하기

id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [13]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02


In [11]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

[<selenium.webdriver.remote.webelement.WebElement (session="6303525807571dd945ab4be62aefd858", element="f.99AE36357ACFAC514C3BE1F7871A1EFC.d.6AF37B53C7EA7B4E61B3C89AF0BB1C37.e.13315")>]

- select 태그 페이지 내에 1개만 존재
- 검색결과 90개로 조정

In [12]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

wait.until(
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

[<selenium.webdriver.remote.webelement.WebElement (session="6303525807571dd945ab4be62aefd858", element="f.99AE36357ACFAC514C3BE1F7871A1EFC.d.6AF37B53C7EA7B4E61B3C89AF0BB1C37.e.13315")>]

In [13]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [17]:
product = products[0]
product.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매\n32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [18]:
## 1번 인덱스 값에서 제품명, 제품스펙, 2번 인덱스 값에서 가격
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')

[<selenium.webdriver.remote.webelement.WebElement (session="c84e346ed17cd31a868dbd3b6df1aa88", element="f.BB0EDD8651D3D588A0077A1346915029.d.F5B34EC3D0BDAA93C8571AA4EF0DC0F0.e.13483")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c84e346ed17cd31a868dbd3b6df1aa88", element="f.BB0EDD8651D3D588A0077A1346915029.d.F5B34EC3D0BDAA93C8571AA4EF0DC0F0.e.13484")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c84e346ed17cd31a868dbd3b6df1aa88", element="f.BB0EDD8651D3D588A0077A1346915029.d.F5B34EC3D0BDAA93C8571AA4EF0DC0F0.e.13485")>]

In [19]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매'

In [20]:
# 컴퓨터 이름 추출
computer_info.find_element(By.CSS_SELECTOR, 'div a'). text

'영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080'

In [21]:
# 컴퓨터 사양 추출
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
' / '.join(spec.text for spec in specs)

'9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용'

In [22]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [23]:
# 가격정보 추출
price_info.find_element(By.CSS_SELECTOR, 'li').text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [30]:
# 인덱스로 데이터 가져오면서 필요값만 필터링
# 원이 나오면 반족 봉료
price_info.find_elements(By.CSS_SELECTOR, 'li > div span')[0].text

'32GB, M.2 1TB'

In [31]:
spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span')

for span in spans:
    if ('위' in span.text): continue
    if ('원' in span.text): break
    print(span.text)

32GB, M.2 1TB
4,529,810


- 각 부분별 동작확인 완료

##### 한 페이지 90개 데이터 스크래핑

In [14]:
import time

# 다나와 게시판 게밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

time.sleep(1.0)

# 총 게시물 읽어오기
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [15]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# 화면에 90개 데이터를 다 조회할때까지 대기
products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
)

# list를 담을 수 있는 배열 생성
danawa_prices = []

# 총 게시물 읽어오기
# len(products)
for index, product in enumerate(products):
    # print(index, product.text)
    items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
    computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
    price_info = items[2]   # 가격정보 들어있는 영역

    # print(computer_info.text)
    ## Pandas DF에 컬럼으로 사용할 이름들 사용
    ## index, computer_name, computer_spec
    computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
    specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
    computer_spec = ' / '.join(spec.text for spec in specs)

    # 가격
    spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
    price_spec = ''
    price = ''

    for span in spans:
        if ('위' in span.text): continue  # 순위 제외
        if ('원' in span.text): break    # 첫번째 금액만 받고 종료

        if ('B' in span.text):
            price_spec = span.text
        else:
            price = span.text

    danawa_prices.append({
        'index': index,
        'computer_name': computer_name,
        'computer_spec': computer_spec,
        'price_spec' : price_spec,
        'price' : price
    })       


In [57]:
len( danawa_prices)

90

In [16]:
## Pandas DF로 변환
df_danawa = pd.DataFrame(danawa_prices)

In [17]:
df_danawa.tail()

,index,computer_name,computer_spec,price_spec,price
85,85,타무즈 게이밍PC TZ330T6HBAA,7800X3D / RTX 5060 / (AMD) B650 / OS미포함 / 800W...,"16GB, M.2 1TB","1,839,220"
86,86,MSI MAG Codex R2 A5NVM I12 U5-225F RTX 5060 Ti...,225F / RTX 5060 Ti / (인텔) H810 / OS미포함 / 600W ...,"16GB, M.2 1TB","1,759,000"
87,87,다나와표준PC DPG 게임용 260222,7800X3D / RTX 5060 Ti / (AMD) B850 / OS미포함 / 7...,"16GB, M.2 1TB","2,638,890"
88,88,STCOM STAI-V138,5600 / RTX 5060 / (AMD) A520 / OS미포함 / 600W / ...,"16GB, M.2 512GB","1,210,710"
89,89,영웅컴퓨터 울트라게이밍 U7277 270K RTX5070,270K Plus / RTX 5070 / (인텔) B860 / OS미포함 / 850...,"32GB, M.2 1TB","3,025,510"


##### 페이지 확인

In [18]:
#
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')
len(buttons)

10

In [19]:
for button in buttons:
    print(button.text)

1
2
3
4
5
6
7
8
9
10


In [20]:
button_nextpage = buttons[9]
button_nextpage.click()

In [21]:
# 다 동작했을 때 10페이지에 있는지 확인
for button in buttons:
    button.click()

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
    )

##### 이전 스크래핑 + 페이지 이동 조합

- 통합 소스 코드

In [22]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# list를 담을 수 있는 배열 생성
danawa_prices = []
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')

# 10개 페이지 이동하면서 스크래핑
for button in buttons:
    button.click()
    page_num = int(button.text)

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
    )

    # 한페이지 스크래핑 내용이 여기 
    # 총 게시물 읽어오기
    # len(products)
    for index, product in enumerate(products):
        # print(index, product.text)
        items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
        computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
        price_info = items[2]   # 가격정보 들어있는 영역

        # print(computer_info.text)
        ## Pandas DF에 컬럼으로 사용할 이름들 사용
        ## index, computer_name, computer_spec
        computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
        specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
        computer_spec = ' / '.join(spec.text for spec in specs)

        # 가격
        spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
        price_spec = ''
        price = ''

        for span in spans:
            if ('위' in span.text): continue  # 순위 제외
            if ('원' in span.text): break    # 첫번째 금액만 받고 종료

            if ('B' in span.text):
                price_spec = span.text
            else:
                price = span.text

        danawa_prices.append({
            'index': ((page_num - 1) * 90) + index,
            'computer_name': computer_name,
            'computer_spec': computer_spec,
            'price_spec' : price_spec,
            'price' : price
        })  

In [23]:
df_total = pd.DataFrame(danawa_prices)

df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   index          900 non-null    int64
 1   computer_name  900 non-null    str  
 2   computer_spec  900 non-null    str  
 3   price_spec     900 non-null    str  
 4   price          900 non-null    str  
dtypes: int64(1), str(4)
memory usage: 35.3 KB


In [24]:
df_total['price']

0      4,529,770
1      1,209,640
2      1,579,200
3      3,943,540
4      2,545,860
         ...    
895    3,400,000
896             
897    9,599,000
898    4,920,000
899    3,095,040
Name: price, Length: 900, dtype: str

In [25]:
# 가격비교예정으로 금액이 없음
df_total[df_total['price'].isna()]

,index,computer_name,computer_spec,price_spec,price


In [26]:
# 가격이 없은 행은 삭제
df_total = df_total.dropna()

In [27]:
# 판다스 컬럼에 천단위 쉼표(,)는 제거후 형변환
df_total['price'] = df_total['price'].str.replace(',', '')

In [28]:
df_total['price'] = pd.to_numeric(df_total['price'])

In [29]:
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          900 non-null    int64  
 1   computer_name  900 non-null    str    
 2   computer_spec  900 non-null    str    
 3   price_spec     900 non-null    str    
 4   price          897 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.3 KB


In [30]:
df_total.to_csv('../data/danawa_게이밍PC.csv', encoding='utf-8', index=False)